# 🎵 Music Mistake Detector — Corrected Experimentation Notebook v2
**Task:** Detect Frequency (F) and Amplitude (A) faults in student music recordings

**Key fixes vs original experimentation_notebook:**
| # | Issue | Fix Applied |
|---|---|---|
| L1 | Threshold tuning on test set | Tuned on **val set only** |
| L2 | Window-level split (overlap leakage) | **Recording-level split** first, then window each split |
| L3 | No student-level holdout | **LOSO** evaluation added as second eval path |
| L4 | Joint F+A model dilutes A gradient | **Two separate models** (F-model, A-model) |
| L5 | Silent augmentation bug | **Fixed**: both X and Y concatenated before shuffle |
| L6 | No feature standardization | **Per-channel z-score** from train statistics only |
| L7 | No confidence intervals | **95% bootstrap CI** on all F1 scores |
| L8 | No rule-based baseline | **Rule-based baseline** added for comparison |
| L9 | Single training run | **3 random seeds**, report mean ± std |
| L10 | CosineAnnealingLR, no plateau | **ReduceLROnPlateau** + early stopping |

**Output folder:** `experimentation_final_2/`


## 🔧 Section 0 — Environment Setup

In [20]:
import subprocess, sys

packages = [
    "torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121",
    "librosa==0.10.2",
    "soundfile",
    "praat-parselmouth",
    "pandas",
    "numpy",
    "scikit-learn",
    "tqdm",
    "scipy",
]

for pkg in packages:
    result = subprocess.run(
        f"{sys.executable} -m pip install {pkg}",
        shell=True, capture_output=True, text=True
    )
    status = '✅' if result.returncode == 0 else '❌'
    print(f"{status}  {pkg.split()[0]}")


✅  torch
✅  librosa==0.10.2
✅  soundfile
✅  praat-parselmouth
✅  pandas
✅  numpy
✅  scikit-learn
✅  tqdm
✅  scipy


In [21]:
import torch, librosa, parselmouth, numpy as np, pandas as pd, sklearn

print(f"torch       : {torch.__version__}")
print(f"librosa     : {librosa.__version__}")
print(f"parselmouth : {parselmouth.__version__}")
print(f"numpy       : {np.__version__}")
print(f"sklearn     : {sklearn.__version__}")

if torch.cuda.is_available():
    print(f"\n✅ GPU : {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️  No GPU — will run on CPU")


torch       : 2.5.1+cu121
librosa     : 0.10.2
parselmouth : 0.4.7
numpy       : 2.2.6
sklearn     : 1.7.2

✅ GPU : NVIDIA RTX A4000
   VRAM: 17.2 GB


## 📂 Section 1 — CONFIG
> **Edit only this cell between experiments.**
> All downstream cells read from CONFIG automatically.

| Setting | Options | Effect |
|---|---|---|
| `split` | `'recording'` / `'loso'` | Recording-level GroupShuffle vs Leave-One-Student-Out |
| `model` | `'crnn'` / `'tcn'` / `'deep_cnn'` | Model architecture |
| `loss` | `'wbce'` / `'focal'` | Loss function |
| `features` | `'base4'` / `'base6'` / `'f_only'` / `'a_only'` | Channel selection |
| `dual_model` | `True` / `False` | Train separate F-model and A-model |
| `augment_A` | `True` / `False` | Include synthetic A-fault aug windows in train |


In [22]:
import os, json, random, warnings
import numpy as np
import torch
warnings.filterwarnings('ignore')

# ============================================================
#   ✏️  EDIT ONLY THIS DICT BETWEEN EXPERIMENTS
# ============================================================
CONFIG = {
    # ── Identity ─────────────────────────────────────────────
    'exp_id':           'E3_focal_fixed',  # unique name for this experiment

    # ── Split ────────────────────────────────────────────────
    # 'recording' = GroupShuffleSplit by file_id (fixes window-overlap leakage)
    # 'loso'      = Leave-One-Student-Out
    'split':            'recording',
    'loso_holdout':     None,   # e.g. 's_2302' (only used when split='loso')
    'val_frac':         0.15,       # fraction of data for validation
    'test_frac':        0.15,       # fraction of data for test

    # ── Dual model (recommended: True) ───────────────────────
    # If True: trains separate F-model and A-model.
    # If False: trains one joint model with 2 output channels.
    'dual_model':      True,


        # ── Task → Feature mapping (5-class) ──────────────────
    # Each task uses only its relevant feature channels
    'task_feature_map': {
        'F'  : 'base6',   # pitch channels only
      #  'K'  : 'base6',    # all 6 channels
      #  'T'  : 'base6',    # all 6 channels
        'AV' : 'base6',   # energy channels only
      #  'SAH': 'base6',    # all 6 channels
    },

    # ── Model ────────────────────────────────────────────────
    'model':            'crnn',  # 'crnn' | 'tcn' | 'deep_cnn'

    # ── Loss ─────────────────────────────────────────────────
    'loss':             'focal',     # 'wbce' | 'focal'
    'focal_alpha':      0.75,
    'focal_gamma':      3.0,

    # ── Features ─────────────────────────────────────────────
    # 'base4'  = [pitch_t, pitch_s, energy_t, energy_s]
    # 'base6'  = base4 + [pitch_diff, energy_diff]
    # 'f_only' = [pitch_t, pitch_s, pitch_diff]          ← F-model input
    # 'a_only' = [energy_t, energy_s, energy_diff]       ← A-model input
    'features':         'base6',

    # ── Augmentation ─────────────────────────────────────────
    'augment_A':        False,      # True = include _aug.npz windows in train

    # ── Training ─────────────────────────────────────────────
    'lr':               1e-3,
    'batch_size':       32,
    'epochs':           120,
    'es_patience':      20,         # early stopping patience
    'lr_patience':      7,          # ReduceLROnPlateau patience
    'lr_factor':        0.5,        # LR reduction factor
    'dropout':          0.3,

    # ── Multi-seed ───────────────────────────────────────────
    # List of seeds → results are mean ± std across seeds
    'seeds':            [42, 1, 2],

    # ── Window config ─────────────────────────────────────────
    'window_size':      400,
    'hop_size':         200,
    'time_step':        0.01,

    # ── Post-processing ───────────────────────────────────────
    'medfilt_kernel':   11,         # median filter kernel (frames) to remove blips

    # ── Bootstrap CI ─────────────────────────────────────────
    'n_bootstrap':      1000,
    'ci_alpha':         0.95,
}
# ============================================================

# ── Paths — derived automatically ──────────────────────────
BASE_DIR = r"C:\Users\RICHIK\Desktop\Monesh\Music_Work\experimentation_final_3_new_annotations"
CSV_PATH    = r"C:\Users\RICHIK\Desktop\Monesh\Music_Work\final_metadata.csv"
NPZ_DIR     = os.path.join(BASE_DIR, "npz_files")
EXP_DIR     = os.path.join(BASE_DIR, "experiments", CONFIG['exp_id'])
SPLITS_DIR  = os.path.join(BASE_DIR, "splits")
MODEL_PATH  = os.path.join(EXP_DIR,  "model_{task}.pt")   # {task} = 'F' or 'A' or 'joint'
RESULTS_PATH= os.path.join(EXP_DIR,  "eval_results.json")
HISTORY_PATH= os.path.join(EXP_DIR,  "training_history.json")

for d in [BASE_DIR, NPZ_DIR, EXP_DIR, SPLITS_DIR]:
    os.makedirs(d, exist_ok=True)

# ── Derived constants ───────────────────────────────────────
WINDOW_SIZE = CONFIG['window_size']
HOP_SIZE    = CONFIG['hop_size']
TIME_STEP   = CONFIG['time_step']
SR          = 44100
HOP_LENGTH  = int(SR * TIME_STEP)
FRAME_LEN   = HOP_LENGTH * 2
DEVICE      = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# ── Channel map ─────────────────────────────────────────────
CHANNEL_MAP = {
    'base4'  : [0, 1, 3, 4],        # pitch_t, pitch_s, energy_t, energy_s
    'base6'  : [0, 1, 2, 3, 4, 5],  # all 6
    'f_only' : [0, 1, 2],           # pitch_t, pitch_s, pitch_diff
    'a_only' : [3, 4, 5],           # energy_t, energy_s, energy_diff
}
N_CHANNELS = len(CHANNEL_MAP[CONFIG['features']])


# ── 5-class constants ────────────────────────────────────
CLASS_NAMES = ['F', 'K', 'T', 'AV', 'SAH']   
CLASS_IDX   = {c: i for i, c in enumerate(CLASS_NAMES)}
# F=0, K=1, T=2, AV=3, SAH=4
N_CLASSES   = 5

# ── Active training tasks (subset of CLASS_NAMES) ───────────
TRAIN_TASKS = list(CONFIG['task_feature_map'].keys())  # ['F', 'AV']

# ── Save config ─────────────────────────────────────────────
with open(os.path.join(EXP_DIR, "config.json"), 'w') as f:
    json.dump(CONFIG, f, indent=2)

print(f"✅ CONFIG loaded: {CONFIG['exp_id']}")
print(f"   Model     : {CONFIG['model']}  |  Loss    : {CONFIG['loss']}")
print(f"   Features  : {CONFIG['features']} ({N_CHANNELS}ch)")
print(f"   Split     : {CONFIG['split']}  |  LOSO : {CONFIG['loso_holdout']}")
print(f"   Dual model: {CONFIG['dual_model']}")
print(f"   Classes   : {CLASS_NAMES}")
print(f"   Task→Feat : {CONFIG['task_feature_map']}")
print(f"   Seeds     : {CONFIG['seeds']}")
print(f"   Device    : {DEVICE}")
print(f"\n   Base dir  : {BASE_DIR}")
print(f"   Exp dir   : {EXP_DIR}")


✅ CONFIG loaded: E3_focal_fixed
   Model     : crnn  |  Loss    : focal
   Features  : base6 (6ch)
   Split     : recording  |  LOSO : None
   Dual model: True
   Classes   : ['F', 'K', 'T', 'AV', 'SAH']
   Task→Feat : {'F': 'base6', 'AV': 'base6'}
   Seeds     : [42, 1, 2]
   Device    : cuda

   Base dir  : C:\Users\RICHIK\Desktop\Monesh\Music_Work\experimentation_final_3_new_annotations
   Exp dir   : C:\Users\RICHIK\Desktop\Monesh\Music_Work\experimentation_final_3_new_annotations\experiments\E3_focal_fixed


## 📊 Section 2 — Load & Verify Dataset

In [23]:
df = pd.read_csv(CSV_PATH)

print(f"✅ CSV loaded!")
print(f"   Shape   : {df.shape}")
print(f"   Columns : {list(df.columns)}")
print(f"\n📊 Per student:")
print(df['student_id'].value_counts())


✅ CSV loaded!
   Shape   : (167, 17)
   Columns : ['annotation_path', 'student_wav_path', 'teacher_wav_path', 'data_fold', 'lesson_id', 'lesson_name', 'teacher_id', 'student_id', 't_scale', 't_bpm', 's_scale', 's_bpm', 'taal', 't_audio', 's_audio', 'anchor_point', 'beat_cycle']

📊 Per student:
student_id
s_2302    72
s_2301    71
s_2303    24
Name: count, dtype: int64


In [24]:
missing = []
for idx, row in df.iterrows():
    for col in ['student_wav_path', 'teacher_wav_path', 'annotation_path']:
        if not os.path.exists(row[col]):
            missing.append(f"Row {idx} | {col} | {os.path.basename(row[col])}")

if len(missing) == 0:
    print(f"✅ All {len(df)*3} files exist!")
else:
    print(f"❌ {len(missing)} files MISSING:")
    for m in missing[:10]:
        print(f"   {m}")


✅ All 501 files exist!


## ⚙️ Section 3 — Feature Extraction Functions
**Design decisions (paper-aligned):**
- `normalize_pitch`: `log2(p/tonic) % 1` — octave-invariant; unvoiced = **−1 sentinel** (NOT interpolated)
- `circular_diff`: wraps at 0/1 boundary — distance(0.05, 0.95)=0.10, not 0.90
- `make_windows`: keeps **frame-level** Y shape `(N, 2, W)` — never collapsed
- 6-channel superset stored in NPZ; `select_channels()` picks subset at load time


In [25]:
from parselmouth.praat import call
from scipy.signal import medfilt

NOTE_TO_FREQ = {
    'C3':130.81,'C#3':138.59,'D3':146.83,'D#3':155.56,'E3':164.81,
    'F3':174.61,'F#3':185.00,'G3':196.00,'G#3':207.65,'A3':220.00,
    'A#3':233.08,'B3':246.94,'C4':261.63,'C#4':277.18,'D4':293.66,
    'D#4':311.13,'E4':329.63,'F4':349.23,'F#4':369.99,'G4':392.00,
    'G#4':415.30,'A4':440.00,'A#4':466.16,'B4':493.88
}


_UNKNOWN_TOKENS_SEEN = set()

def get_classes(token):
    t = token.strip()
    if t == "F-TV":                              return {"F", "T", "AV"}
    if t == "F-A":                               return {"F", "AV"}
    if t in ("F-SA", "F- SA"):                   return {"F", "SAH"}
    if t in ("F", "F1", "F-M", "F- Trembling voice"): return {"F"}
    if t == "K":                                 return {"K"}
    if t == "T":                                 return {"T"}
    if t in ("A", "V"):                          return {"AV"}
    if t in ("SA", "H", "H- holding notes"):     return {"SAH"}
    if t and t not in _UNKNOWN_TOKENS_SEEN:
        print(f"  ⚠️  Unknown token skipped: '{t}'")
        _UNKNOWN_TOKENS_SEEN.add(t)
    return set()

def parse_annotation(ann_path):
    result = []
    with open(ann_path, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            parts = line.strip().split('\t')
            if len(parts) < 3:
                continue
            try:
                start = float(parts[0])
                end   = float(parts[1])
            except ValueError:
                continue
            tokens = parts[2].split(';')
            row_classes = set()
            for tok in tokens:
                row_classes.update(get_classes(tok.strip()))
            if row_classes:
                result.append({'start': start, 'end': end, 'classes': row_classes})
    return result



def make_frame_labels(ann_path, audio_duration, time_step=TIME_STEP):
    n_frames = int(np.ceil(audio_duration / time_step))
    Y = np.zeros((5, n_frames), dtype=np.float32)  # (5, N) — F,K,T,AV,SAH
    for seg in parse_annotation(ann_path):
        s = max(0, int(np.floor(seg['start'] / time_step)))
        e = min(n_frames, int(np.ceil(seg['end']  / time_step)))
        for cls in seg['classes']:
            Y[CLASS_IDX[cls], s:e] = 1.0
    return Y  # shape: (5, N)


def extract_pitch(audio_path, time_step=TIME_STEP):
    """Praat pitch → Hz (0 = unvoiced, NOT NaN)."""
    sound  = parselmouth.Sound(audio_path)
    pitch  = sound.to_pitch(time_step=time_step,
                             pitch_floor=75.0, pitch_ceiling=600.0)
    hz     = pitch.selected_array['frequency'].astype(np.float32)
    return hz   # 0 = unvoiced; normalize_pitch handles sentinel


def normalize_pitch(pitch_hz, tonic_note):
    """
    Paper Eq.1: p̂ = log2(p/tonic) % 1  for voiced frames.
    Unvoiced (pitch==0) → sentinel −1.
    No fill_nans — the −1 carries real information ('student is silent here').
    """
    tonic  = NOTE_TO_FREQ[tonic_note]
    out    = np.full_like(pitch_hz, -1.0, dtype=np.float32)
    voiced = pitch_hz > 0
    out[voiced] = np.mod(np.log2(pitch_hz[voiced] / tonic), 1.0).astype(np.float32)
    return out


def extract_energy(audio_path, sr=SR, hop_length=HOP_LENGTH):
    y, _ = librosa.load(audio_path, sr=sr)
    rms   = librosa.feature.rms(y=y, frame_length=FRAME_LEN,
                                 hop_length=hop_length)[0]
    return np.log10(rms + 1e-10).astype(np.float32)


def align_to_length(arr, target_len):
    if len(arr) == target_len:
        return arr
    return np.interp(np.linspace(0, 1, target_len),
                     np.linspace(0, 1, len(arr)), arr).astype(np.float32)


def circular_diff(a, b):
    """
    Circular distance in [0,1) pitch space.
    Unvoiced frames (−1) → 0 difference.
    """
    mask = (a < 0) | (b < 0)
    d    = a - b
    d    = np.where(np.abs(d) > 0.5, np.sign(d) * (np.abs(d) - 1.0), d)
    d[mask] = 0.0
    return d.astype(np.float32)


def select_channels(X_6ch, feature_key):
    """X_6ch: (6, N) superset → select subset by feature key."""
    return X_6ch[CHANNEL_MAP[feature_key], :]


def make_windows(X, Y, window_size=WINDOW_SIZE, hop_size=HOP_SIZE):
    """
    X : (C, N)  →  (num_windows, C, W)
    Y : (2, N)  →  (num_windows, 2, W)   frame-level, never collapsed
    """
    X_wins, Y_wins = [], []
    N = X.shape[1]
    for start in range(0, N - window_size + 1, hop_size):
        end = start + window_size
        X_wins.append(X[:, start:end])
        Y_wins.append(Y[:, start:end])
    return np.array(X_wins, dtype=np.float32), np.array(Y_wins, dtype=np.float32)


print("✅ Feature functions defined")
print("   normalize_pitch : log2 % 1  |  unvoiced = −1 (no interpolation)")
print("   circular_diff   : octave-wrap-aware pitch delta")
print("   make_windows : frame-level Y (N, 5, W)  [F,K,T,AV,SAH]")


✅ Feature functions defined
   normalize_pitch : log2 % 1  |  unvoiced = −1 (no interpolation)
   circular_diff   : octave-wrap-aware pitch delta
   make_windows : frame-level Y (N, 5, W)  [F,K,T,AV,SAH]


## 💾 Section 4 — Process Full Dataset → Save .npz
> Always extracts **6-channel superset** (`select_channels()` picks subset later).  
> Skip-if-exists: delete `npz_files/` to force re-extraction.  
> Augmented files (`_aug.npz`) are created unconditionally so all experiments share them.


In [26]:
from tqdm import tqdm
import soundfile as sf, tempfile

# ── Skip-if-exists ──────────────────────────────────────────
existing = [f for f in os.listdir(NPZ_DIR) if f.endswith('.npz') and '_aug' not in f]
if len(existing) == len(df):
    print(f"✅ NPZ files already exist ({len(df)}) — skipping extraction")
    print(f"   Delete {NPZ_DIR} to force re-extraction.")
else:
    print(f"📦 Extracting 6-channel features for {len(df)} recordings...")
    errors, saved = [], 0
    rng = np.random.default_rng(42)

    for idx, row in tqdm(df.iterrows(), total=len(df), desc="Extracting"):
        try:
            # ── Pitch ─────────────────────────────────────
            pitch_s = normalize_pitch(extract_pitch(row['student_wav_path']),  row['s_scale'])
            pitch_t = normalize_pitch(extract_pitch(row['teacher_wav_path']),  row['t_scale'])

            # ── Energy ────────────────────────────────────
            energy_s = extract_energy(row['student_wav_path'])
            energy_t = extract_energy(row['teacher_wav_path'])

            # ── Align to min length ────────────────────────
            tlen = min(len(pitch_s), len(pitch_t), len(energy_s), len(energy_t))
            p_t  = align_to_length(pitch_t,  tlen)
            p_s  = align_to_length(pitch_s,  tlen)
            e_t  = align_to_length(energy_t, tlen)
            e_s  = align_to_length(energy_s, tlen)

            # ── 6-channel superset ─────────────────────────
            # 0=pitch_t  1=pitch_s  2=pitch_diff
            # 3=energy_t 4=energy_s 5=energy_diff
            X = np.stack([p_t, p_s, circular_diff(p_s, p_t),
                          e_t, e_s, e_s - e_t], axis=0)  # (6, N)

            # ── Frame-level labels ─────────────────────────
            Y_raw = make_frame_labels(row['annotation_path'],
                                    audio_duration=tlen * TIME_STEP)
            Y = np.stack([
                align_to_length(Y_raw[i], tlen).round().astype(np.float32)
                for i in range(5)
            ], axis=0)  # (5, N)  — F,K,T,AV,SAH            

            np.savez_compressed(
                os.path.join(NPZ_DIR, f"sample_{idx:03d}.npz"),
                X=X, Y=Y,
                student_id=str(row['student_id']),
                lesson=str(row['lesson_name']),
                file_id=str(idx)
            )
            saved += 1

            # ── Synthetic A-fault augmentation (Algorithm 1) ──────────────
            # Created unconditionally (used only when CONFIG['augment_A']=True)
            try:
                y_wave, _ = librosa.load(row['student_wav_path'], sr=SR)
                snd       = parselmouth.Sound(row['student_wav_path'])
                pitch_obj = snd.to_pitch(time_step=TIME_STEP,
                                          pitch_floor=75, pitch_ceiling=600)
                p_hz = pitch_obj.selected_array['frequency'].astype(np.float64)
                p_hz[p_hz <= 0] = 1e-6
                p_hz  = medfilt(p_hz, kernel_size=21)
                midi  = 69 + 12 * np.log2(p_hz / 440.0)

                # Segment stable notes
                segs, n0 = [], 0
                for n in range(1, len(midi)):
                    if abs(midi[n] - midi[n0]) > 0.5:
                        if (n - n0) * TIME_STEP >= 0.5:
                            segs.append((n0, n))
                        n0 = n
                if (len(midi) - n0) * TIME_STEP >= 0.5:
                    segs.append((n0, len(midi)))

                static = []
                for s, e in segs:
                    ss, es = int(s*TIME_STEP*SR), int(e*TIME_STEP*SR)
                    seg = y_wave[ss:es]
                    if len(seg) == 0: continue
                    rms_val = np.sqrt(np.mean(seg**2))
                    sigma   = np.std(midi[s:e][15:-15]) if (e-s) > 30 else np.std(midi[s:e])
                    if rms_val >= 0.01 and sigma <= 10.0:
                        static.append((s, e, ss, es))

                if len(static) == 0:
                    raise ValueError("No stable notes found for augmentation")

                y_aug  = y_wave.copy()
                a_lbl  = np.zeros(len(midi), dtype=np.float32)
                n_pick = max(1, len(static) // 2)
                picks  = rng.choice(len(static), size=min(n_pick, len(static)), replace=False)

                for i in picks:
                    s, e, ss, es = static[i]
                    r = float(rng.uniform(0.2, 0.6) if rng.random() < 0.5
                               else rng.uniform(1.2, 1.8))
                    y_aug[ss:es] = (y_aug[ss:es] * r).astype(np.float32)
                    a_lbl[s:e]   = 1.0

                # Re-extract pitch/energy from perturbed audio
                with tempfile.NamedTemporaryFile(suffix='.wav', delete=False) as tmp:
                    sf.write(tmp.name, y_aug, SR)
                    tmp_path = tmp.name

                p_s_aug  = normalize_pitch(extract_pitch(tmp_path), row['s_scale'])
                e_s_aug  = extract_energy(tmp_path)
                os.unlink(tmp_path)

                p_s_aug_a  = align_to_length(p_s_aug,  tlen)
                e_s_aug_a  = align_to_length(e_s_aug,  tlen)
                a_lbl_a    = align_to_length(a_lbl, tlen).round().astype(np.float32)

                X_aug = np.stack([p_t, p_s_aug_a, circular_diff(p_s_aug_a, p_t),
                                   e_t, e_s_aug_a, e_s_aug_a - e_t], axis=0)
 
                Y_aug = Y.copy()
                Y_aug[3] = np.clip(Y[3] + a_lbl_a, 0, 1).astype(np.float32)
                # index 3 = AV (amplitude/volume fault)

                np.savez_compressed(
                    os.path.join(NPZ_DIR, f"sample_{idx:03d}_aug.npz"),
                    X=X_aug, Y=Y_aug,
                    student_id=str(row['student_id']),
                    lesson=str(row['lesson_name']),
                    file_id=str(idx)
                )
            except Exception:
                pass  # Aug failure is non-fatal

        except Exception as e:
            errors.append({'idx': idx, 'error': str(e)})

    print(f"\n✅ Saved   : {saved} / {len(df)}")
    print(f"❌ Errors  : {len(errors)}")

base_files = sorted(f for f in os.listdir(NPZ_DIR)
                    if f.endswith('.npz') and '_aug' not in f)
aug_files  = sorted(f for f in os.listdir(NPZ_DIR) if f.endswith('_aug.npz'))
print(f"\n📊 NPZ: base={len(base_files)}  aug={len(aug_files)}")


✅ NPZ files already exist (167) — skipping extraction
   Delete C:\Users\RICHIK\Desktop\Monesh\Music_Work\experimentation_final_3_new_annotations\npz_files to force re-extraction.

📊 NPZ: base=167  aug=167


In [27]:
# ── 🔍 DIAGNOSTIC — verify NPZ shapes & 5-class labels ──────
import random

base_files = sorted(f for f in os.listdir(NPZ_DIR)
                    if f.endswith('.npz') and '_aug' not in f)

sample_files = random.sample(base_files, min(3, len(base_files)))

for fname in sample_files:
    s = np.load(os.path.join(NPZ_DIR, fname), allow_pickle=True)
    Y = s['Y']  # should be (5, N)
    print(f"\n📦 {fname}")
    print(f"   X.shape = {s['X'].shape}  |  Y.shape = {Y.shape}")
    assert Y.shape[0] == 5, f"❌ Y has {Y.shape[0]} channels — expected 5!"
    for i, cls in enumerate(CLASS_NAMES):
        pct = float(Y[i].mean()) * 100
        print(f"   {cls:<4}: {pct:5.2f}% positive frames")

print("\n✅ Diagnostic passed — all sampled NPZs have Y shape (5, N)")


📦 sample_014.npz
   X.shape = (6, 2461)  |  Y.shape = (5, 2461)
   F   : 29.30% positive frames
   K   :  0.00% positive frames
   T   :  0.00% positive frames
   AV  :  8.66% positive frames
   SAH :  9.59% positive frames

📦 sample_023.npz
   X.shape = (6, 4013)  |  Y.shape = (5, 4013)
   F   : 20.03% positive frames
   K   :  3.46% positive frames
   T   :  0.00% positive frames
   AV  :  0.00% positive frames
   SAH :  0.00% positive frames

📦 sample_021.npz
   X.shape = (6, 2547)  |  Y.shape = (5, 2547)
   F   : 43.97% positive frames
   K   :  0.00% positive frames
   T   :  0.00% positive frames
   AV  :  0.00% positive frames
   SAH : 12.29% positive frames

✅ Diagnostic passed — all sampled NPZs have Y shape (5, N)


## 🪟 Section 5 — Recording-Level Split → Then Window Each Split
> **This is the key leakage fix.**  
> Old approach: window all → split windows randomly → overlapping windows in both train and test.  
> New approach: split recordings first → window each split independently → zero overlap guaranteed.

### Split modes
- `'recording'`: `GroupShuffleSplit` with `groups=file_id` → 70/15/15
- `'loso'`: one student's recordings → test; 85% of rest → train; 15% of rest → val


In [28]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from collections import Counter

base_files = sorted(f for f in os.listdir(NPZ_DIR)
                    if f.endswith('.npz') and '_aug' not in f)
aug_files  = sorted(f for f in os.listdir(NPZ_DIR) if f.endswith('_aug.npz'))

# ── Build metadata arrays for all base files ─────────────────────────────
all_fnames, all_student_ids = [], []
for fname in base_files:
    s = np.load(os.path.join(NPZ_DIR, fname), allow_pickle=True)
    all_fnames.append(fname)
    all_student_ids.append(str(s['student_id']))

all_fnames      = np.array(all_fnames)
all_student_ids = np.array(all_student_ids)
file_indices    = np.arange(len(all_fnames))

print(f"Total recordings : {len(all_fnames)}")
print("Per student:", dict(Counter(all_student_ids)))

def merge_rare(labels, min_count=2):
    counts = Counter(labels)
    merged = [lbl if counts[lbl] >= min_count else "rare" for lbl in labels]
    # If "rare" bucket itself has < min_count, fold it into the majority class
    merged_counts = Counter(merged)
    if merged_counts.get("rare", 0) < min_count:
        majority = [k for k, _ in merged_counts.most_common() if k != "rare"][0]
        merged = [majority if lbl == "rare" else lbl for lbl in merged]
    return merged

# ── Choose split strategy ─────────────────────────────────────────────────
if CONFIG['split'] == 'recording':
    SPLITS_PATH = os.path.join(SPLITS_DIR, "recording_split_filenames.json")

    if os.path.exists(SPLITS_PATH):
        print(f"\n✅ Loading existing recording-level split from {SPLITS_PATH}")
        with open(SPLITS_PATH) as f:
            split_meta = json.load(f)
        train_fnames = split_meta['train']
        val_fnames   = split_meta['val']
        test_fnames  = split_meta['test']

    else:
        # Build per-recording label: "00", "10", "01", "11"
        rec_labels = []
        for fname in all_fnames:
            s = np.load(os.path.join(NPZ_DIR, fname), allow_pickle=True)
            has = ''.join(['1' if s['Y'][i].max() > 0 else '0' for i in range(5)])
            student = str(s['student_id'])
            rec_labels.append(f"{student}_{has}")            

        # Merge any label with count < 2 into "rare" so stratify doesn't crash
        rec_labels_safe = merge_rare(rec_labels)

        print("\nLabel distribution (after rare-merge):")
        print(dict(Counter(rec_labels_safe)))

        # Step 1: carve out test set
        tv_idx, test_idx = train_test_split(
            file_indices,
            test_size=CONFIG['test_frac'],
            stratify=rec_labels_safe,
            random_state=42
        )

        # Step 2: carve out val from remainder
        val_frac_of_tv = CONFIG['val_frac'] / (1.0 - CONFIG['test_frac'])
        rec_labels_tv  = [rec_labels_safe[i] for i in tv_idx]

        # Same rare-merge for the tv subset
        rec_labels_tv_safe = merge_rare([rec_labels_safe[i] for i in tv_idx])

        train_sub_idx, val_sub_idx = train_test_split(
            tv_idx,
            test_size=val_frac_of_tv,
            stratify=rec_labels_tv_safe,
            random_state=42
        )

        train_fnames = list(all_fnames[train_sub_idx])
        val_fnames   = list(all_fnames[val_sub_idx])
        test_fnames  = list(all_fnames[test_idx])

        split_meta = {'train': train_fnames, 'val': val_fnames, 'test': test_fnames}
        with open(SPLITS_PATH, 'w') as f:
            json.dump(split_meta, f, indent=2)
        print(f"\n💾 Saved recording-level split → {SPLITS_PATH}")

    print(f"\n✅ Recording-level split:")
    print(f"   Train: {len(train_fnames)} recordings")
    print(f"   Val  : {len(val_fnames)} recordings")
    print(f"   Test : {len(test_fnames)} recordings")

    # Verify all 3 students in test
    test_students = set(all_student_ids[i] for i, f in enumerate(all_fnames)
                        if f in set(test_fnames))
    print(f"   Students in test: {sorted(test_students)}  ← should be all 3")

elif CONFIG['split'] == 'loso':
    held_out = CONFIG['loso_holdout']
    assert held_out is not None, "Set CONFIG['loso_holdout'] to a student id!"

    SPLITS_PATH = os.path.join(SPLITS_DIR, f"loso_split_{held_out}_filenames.json")

    if os.path.exists(SPLITS_PATH):
        print(f"\n✅ Loading existing LOSO split ({held_out}) from {SPLITS_PATH}")
        with open(SPLITS_PATH) as f:
            split_meta = json.load(f)
        train_fnames = split_meta['train']
        val_fnames   = split_meta['val']
        test_fnames  = split_meta['test']
    else:
        test_mask  = all_student_ids == held_out
        non_test   = all_fnames[~test_mask]
        non_test_s = all_student_ids[~test_mask]

        gss_val = GroupShuffleSplit(n_splits=1, test_size=0.15, random_state=42)
        tr_sub, v_sub = next(gss_val.split(np.arange(len(non_test)),
                                            groups=non_test_s))
        train_fnames = list(non_test[tr_sub])
        val_fnames   = list(non_test[v_sub])
        test_fnames  = list(all_fnames[test_mask])

        split_meta = {'train': train_fnames, 'val': val_fnames, 'test': test_fnames}
        with open(SPLITS_PATH, 'w') as f:
            json.dump(split_meta, f, indent=2)
        print(f"\n💾 Saved LOSO split → {SPLITS_PATH}")

    print(f"\n✅ LOSO split (held-out: {held_out}):")
    print(f"   Train: {len(train_fnames)} recordings")
    print(f"   Val  : {len(val_fnames)} recordings")
    print(f"   Test : {len(test_fnames)} recordings")

else:
    raise ValueError(f"Unknown split: {CONFIG['split']}")

Total recordings : 167
Per student: {np.str_('s_2301'): 71, np.str_('s_2302'): 72, np.str_('s_2303'): 24}

✅ Loading existing recording-level split from C:\Users\RICHIK\Desktop\Monesh\Music_Work\experimentation_final_3_new_annotations\splits\recording_split_filenames.json

✅ Recording-level split:
   Train: 116 recordings
   Val  : 25 recordings
   Test : 26 recordings
   Students in test: [np.str_('s_2301'), np.str_('s_2302'), np.str_('s_2303')]  ← should be all 3


## 🪟 Section 6 — Windowing (per split, no cross-contamination)

In [29]:
def load_and_window(fname_list, feature_key, include_aug=False, desc=""):
    """
    Load NPZ files, select channels, window.
    Returns X (N, C, W), Y (N, 2, W).
    Windows ONLY come from recordings in fname_list → no cross-split overlap.
    """
    X_list, Y_list = [], []
    for fname in tqdm(fname_list, desc=desc or "Windowing"):
        s     = np.load(os.path.join(NPZ_DIR, fname), allow_pickle=True)
        X_sel = select_channels(s['X'], feature_key)
        X_w, Y_w = make_windows(X_sel, s['Y'])
        if len(X_w) == 0:
            continue
        X_list.append(X_w)
        Y_list.append(Y_w)

    if include_aug:
        # Only augmented versions of TRAINING files
        for fname in tqdm(fname_list, desc="Windowing aug"):
            aug_fname = fname.replace('.npz', '_aug.npz')
            aug_path  = os.path.join(NPZ_DIR, aug_fname)
            if not os.path.exists(aug_path):
                continue
            s     = np.load(aug_path, allow_pickle=True)
            X_sel = select_channels(s['X'], feature_key)
            X_w, Y_w = make_windows(X_sel, s['Y'])
            if len(X_w) == 0:
                continue
            X_list.append(X_w)
            Y_list.append(Y_w)

    if not X_list:
        return np.empty((0,)), np.empty((0,))

    X = np.concatenate(X_list, axis=0)
    Y = np.concatenate(Y_list, axis=0)

    # Shuffle train (important after augmentation)
    rng = np.random.default_rng(42)   # fixed seed — same order every call
    idx = rng.permutation(len(X))
    return X[idx], Y[idx]


print("📂 Windowing per task...")
TASK_WINDOWS = {}

for task, feat_key in CONFIG['task_feature_map'].items():
    X_tr, Y_tr = load_and_window(train_fnames, feat_key,
                                  include_aug=(CONFIG['augment_A'] and task == 'AV'),
                                  desc=f"Train-{task}")
    X_v,  Y_v  = load_and_window(val_fnames,  feat_key,
                                  include_aug=False, desc=f"Val-{task}")
    X_te, Y_te = load_and_window(test_fnames, feat_key,
                                  include_aug=False, desc=f"Test-{task}")
    TASK_WINDOWS[task] = {
        'feat': feat_key,
        'Xtr': X_tr, 'Ytr': Y_tr,
        'Xv' : X_v,  'Yv' : Y_v,
        'Xte': X_te, 'Yte': Y_te,
    }
    cls_i  = CLASS_IDX[task]
    pct_tr = float(Y_tr[:, cls_i, :].mean()) * 100 if len(Y_tr) else 0
    print(f"  {task:<4}: train={len(X_tr):,} windows | {task}_positive={pct_tr:.2f}%")

print(f"\n✅ TASK_WINDOWS ready — tasks: {list(TASK_WINDOWS.keys())}")


📂 Windowing per task...


Test-F: 100%|██████████| 26/26 [00:00<00:00, 1039.55it/s]


  F   : train=2,776 windows | F_positive=33.75%


Test-AV: 100%|██████████| 26/26 [00:00<00:00, 1130.45it/s]

  AV  : train=2,776 windows | AV_positive=3.42%

✅ TASK_WINDOWS ready — tasks: ['F', 'AV']


## 📐 Section 7 — Feature Standardization
> **Fit on training data only.** Apply the same statistics to val and test.  
> This prevents any statistical information from val/test leaking into the model.  
> Channels that use −1 as a sentinel (pitch channels 0, 1, 2) are standardized  
> only on voiced frames; the −1 sentinel is preserved.


In [30]:
class ChannelStandardizer:
    """
    Per-channel z-score normalization.
    Pitch channels (where values can be -1 = unvoiced) are fitted on voiced frames only.
    The -1 sentinel is preserved through normalization (not shifted/scaled).
    """
    def __init__(self, pitch_channel_indices=None):
        self.pitch_channels = set(pitch_channel_indices or [])
        self.mean_ = None
        self.std_  = None

    def fit(self, X):
        """X: (N, C, W) — fit statistics on training data only."""
        N, C, W = X.shape
        self.mean_ = np.zeros(C, dtype=np.float32)
        self.std_  = np.ones( C, dtype=np.float32)
        for c in range(C):
            vals = X[:, c, :].reshape(-1)
            if c in self.pitch_channels:
                voiced = vals[vals >= 0]  # exclude -1 sentinel
                if len(voiced) > 0:
                    self.mean_[c] = voiced.mean()
                    self.std_[c]  = voiced.std() + 1e-8
            else:
                self.mean_[c] = vals.mean()
                self.std_[c]  = vals.std() + 1e-8
        return self

    def transform(self, X):
        """X: (N, C, W) — apply stored statistics."""
        X_out = X.copy()
        for c in range(X.shape[1]):
            if c in self.pitch_channels:
                # Only transform voiced frames; keep -1 as-is
                mask = X_out[:, c, :] >= 0
                X_out[:, c, :] = np.where(
                    mask,
                    (X_out[:, c, :] - self.mean_[c]) / self.std_[c],
                    X_out[:, c, :]   # keep -1 sentinel unchanged
                )
            else:
                X_out[:, c, :] = (X_out[:, c, :] - self.mean_[c]) / self.std_[c]
        return X_out

    def fit_transform(self, X):
        return self.fit(X).transform(X)

    def save(self, path):
        np.savez(path, mean=self.mean_, std=self.std_)

    @classmethod
    def load(cls, path, pitch_channel_indices=None):
        d = np.load(path)
        s = cls(pitch_channel_indices)
        s.mean_, s.std_ = d['mean'], d['std']
        return s


# ── Identify pitch channels for the current feature config ────────────────
# In base6: channels 0,1,2 are pitch-related (pitch_t, pitch_s, pitch_diff)
# In base4: channels 0,1 are pitch-related
# In f_only: channels 0,1,2 are all pitch
# In a_only: no pitch channels
PITCH_CH_MAP = {
    'base4' : [0, 1],
    'base6' : [0, 1],
    'f_only': [0, 1],
    'a_only': [],
}

TASK_SCALERS = {}

for task in TRAIN_TASKS:
    feat_key  = CONFIG['task_feature_map'][task]
    pitch_chs = PITCH_CH_MAP[feat_key]
    sc = ChannelStandardizer(pitch_channel_indices=pitch_chs)
    w  = TASK_WINDOWS[task]
    w['Xtr'] = sc.fit_transform(w['Xtr'])
    w['Xv']  = sc.transform(w['Xv'])
    w['Xte'] = sc.transform(w['Xte'])
    TASK_SCALERS[task] = sc
    print(f"  ✅ {task:<4} ({feat_key}) — pitch_chs={pitch_chs}")

# Save all scaler stats
np.savez(
    os.path.join(EXP_DIR, "scaler_stats.npz"),
    **{f"{t}_mean": TASK_SCALERS[t].mean_ for t in TRAIN_TASKS},
    **{f"{t}_std" : TASK_SCALERS[t].std_  for t in TRAIN_TASKS},
)

print("\n✅ Feature standardization complete — per-task scalers fitted on train only")


  ✅ F    (base6) — pitch_chs=[0, 1]
  ✅ AV   (base6) — pitch_chs=[0, 1]

✅ Feature standardization complete — per-task scalers fitted on train only


## 🏗️ Section 8 — Model & Loss Factories
> **Dual-model design** (when `CONFIG['dual_model']=True`):  
> - F-model: input = pitch channels only (`f_only` config) — dedicated to frequency fault  
> - A-model: input = energy channels only (`a_only` config) — dedicated to amplitude fault  
>  
> This prevents the A gradient (rare, hard) from being diluted by the F gradient.  
> Each model is smaller and more focused.


In [31]:
import torch.nn as nn, torch.nn.functional as F_
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset


# ══════════════════════════════════════════════════════════════
#   MODEL DEFINITIONS
# ══════════════════════════════════════════════════════════════

class CRNN(nn.Module):
    """
    3×Conv1d + BiGRU + linear head.
    Appropriate size (~5K params) for a small dataset.
    Input : (B, C, W)  →  Output : (B, out_ch, W)
    """
    def __init__(self, in_ch, out_ch=1, hidden=16, dropout=0.3):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv1d(in_ch, 8,  3, padding=1), nn.BatchNorm1d(8),  nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(8,    16,  3, padding=1), nn.BatchNorm1d(16), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(16,    8,  3, padding=1), nn.BatchNorm1d(8),  nn.ReLU(), nn.Dropout(dropout),
        )
        self.gru  = nn.GRU(8, hidden, batch_first=True, bidirectional=True)
        self.head = nn.Linear(2 * hidden, out_ch)

    def forward(self, x):
        z = self.conv(x).transpose(1, 2)
        z, _ = self.gru(z)
        return torch.sigmoid(self.head(z).transpose(1, 2))


class TCNBlock(nn.Module):
    def __init__(self, in_c, out_c, dilation, dropout=0.2):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_c, out_c, 3, padding=dilation, dilation=dilation),
            nn.BatchNorm1d(out_c), nn.ReLU(), nn.Dropout(dropout))
    def forward(self, x): return self.net(x)


class TCN(nn.Module):
    """
    Encoder-decoder TCN with dilated convolutions (~12K params).
    """
    def __init__(self, in_ch, out_ch=1, dropout=0.2):
        super().__init__()
        self.e1   = TCNBlock(in_ch, 8,  dilation=1, dropout=dropout)
        self.e2   = TCNBlock(8,    16,  dilation=2, dropout=dropout)
        self.e3   = TCNBlock(16,   32,  dilation=4, dropout=dropout)
        self.pool = nn.MaxPool1d(2)
        self.up   = nn.Upsample(scale_factor=2, mode='linear', align_corners=False)
        self.d3   = TCNBlock(32, 32, dilation=4, dropout=dropout)
        self.d2   = TCNBlock(32, 16, dilation=2, dropout=dropout)
        self.d1   = TCNBlock(16,  8, dilation=1, dropout=dropout)
        self.head = nn.Conv1d(8, out_ch, 3, padding=1)

    def forward(self, x):
        x = self.pool(self.e1(x))
        x = self.pool(self.e2(x))
        x = self.e3(x)
        x = self.up(self.d3(x))
        x = self.up(self.d2(x))
        x = self.d1(x)
        return torch.sigmoid(self.head(x))


class DeepCNN(nn.Module):
    """
    Deep CNN (~125K params). Likely to overfit on 167 recordings.
    Included for comparison — use with strong dropout.
    """
    def __init__(self, in_ch, out_ch=1, dropout=0.3):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv1d(in_ch,  32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(32,     64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64,     64, 3, padding=1), nn.BatchNorm1d(64), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(64,     32, 3, padding=1), nn.BatchNorm1d(32), nn.ReLU(), nn.Dropout(dropout),
            nn.Conv1d(32, out_ch, 3, padding=1), nn.Sigmoid()
        )
    def forward(self, x): return self.net(x)


def build_model(cfg, n_ch, out_ch=1):
    m, d = cfg['model'], cfg['dropout']
    if   m == 'crnn':     return CRNN(in_ch=n_ch,     out_ch=out_ch, dropout=d)
    elif m == 'tcn':      return TCN(in_ch=n_ch,      out_ch=out_ch, dropout=d)
    elif m == 'deep_cnn': return DeepCNN(in_ch=n_ch,  out_ch=out_ch, dropout=d)
    else: raise ValueError(f"Unknown model: {m}")


# ══════════════════════════════════════════════════════════════
#   LOSS DEFINITIONS
# ══════════════════════════════════════════════════════════════

class WeightedBCELoss(nn.Module):
    """
    Paper's inverse-class-frequency weighted BCE.
    w1 = 0.5/pos_freq,  w0 = 0.5/neg_freq
    Computed from training data only.
    """
    def __init__(self, pos_freq):
        super().__init__()
        p = max(pos_freq, 1e-4)
        self.w1 = 0.5 / p
        self.w0 = 0.5 / (1.0 - p)

    def forward(self, pred, target):
        eps  = 1e-7
        pred = pred.clamp(eps, 1 - eps)
        loss = -(self.w1 * target * torch.log(pred) +
                 self.w0 * (1 - target) * torch.log(1 - pred))
        return loss.mean()


class FocalLoss(nn.Module):
    def __init__(self, alpha=0.75, gamma=3.0):
        super().__init__()
        self.alpha, self.gamma = alpha, gamma

    def forward(self, pred, target):
        eps  = 1e-7
        pred = pred.clamp(eps, 1 - eps)
        bce  = -(target * torch.log(pred) + (1 - target) * torch.log(1 - pred))
        pt   = torch.where(target == 1, pred, 1 - pred)
        return (self.alpha * (1 - pt) ** self.gamma * bce).mean()


def build_loss(cfg, pos_freq):
    if   cfg['loss'] == 'wbce':  return WeightedBCELoss(pos_freq)
    elif cfg['loss'] == 'focal': return FocalLoss(cfg['focal_alpha'], cfg['focal_gamma'])
    else: raise ValueError(f"Unknown loss: {cfg['loss']}")


# ── Quick sanity check ─────────────────────────────────────────────────────
_n_ch_check = len(CHANNEL_MAP[CONFIG['task_feature_map']['F']])  # F-task ke liye check
_m = build_model(CONFIG, _n_ch_check, out_ch=1)
_d = torch.randn(4, _n_ch_check, WINDOW_SIZE)
with torch.no_grad():
    _o = _m(_d)
print(f"✅ {CONFIG['model']} | in={_d.shape} → out={_o.shape} | params={sum(p.numel() for p in _m.parameters()):,}")


✅ crnn | in=torch.Size([4, 6, 400]) → out=torch.Size([4, 1, 400]) | params=3,537


## 🚀 Section 9 — Training
> **Dual-model mode** (recommended): trains a separate F-model and A-model.  
> Each model receives only its relevant channels and has its own class-weighted loss.  
> **3 seeds** → results are mean ± std to show stability.  
> **ReduceLROnPlateau** + **early stopping** on val loss.


In [32]:
from scipy.signal import medfilt as scipy_medfilt


def make_loader(X, Y_1ch, batch_size=CONFIG['batch_size'], shuffle=True):
    """
    X     : (N, C, W)
    Y_1ch : (N, 1, W) or (N, W) — single-channel frame labels
    """
    Xt = torch.tensor(X, dtype=torch.float32)
    if Y_1ch.ndim == 2:
        Yt = torch.tensor(Y_1ch[:, None, :], dtype=torch.float32)
    else:
        Yt = torch.tensor(Y_1ch, dtype=torch.float32)
    return DataLoader(TensorDataset(Xt, Yt), batch_size=batch_size,
                      shuffle=shuffle, num_workers=0, pin_memory=True)


def train_one_model(X_tr, Y_tr_1ch, X_v, Y_v_1ch,
                    n_ch, pos_freq, task_name, seed, cfg):
    """
    Train a single-output model for one task (F or A) with one seed.
    Returns: best checkpoint path, training history dict, best val loss.
    """
    torch.manual_seed(seed)
    np.random.seed(seed)
    random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    model     = build_model(cfg, n_ch, out_ch=1).to(DEVICE)
    criterion = build_loss(cfg, pos_freq)
    optimizer = optim.Adam(model.parameters(), lr=cfg['lr'])
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='min',
        patience=cfg['lr_patience'],
        factor=cfg['lr_factor'],
        verbose=False
    )

    tr_loader = make_loader(X_tr, Y_tr_1ch, shuffle=True)
    v_loader  = make_loader(X_v,  Y_v_1ch,  shuffle=False)

    ckpt_path     = os.path.join(EXP_DIR, f"model_{task_name}_seed{seed}.pt")
    best_val_loss = float('inf')
    bad_epochs    = 0
    best_epoch    = 0
    history       = {'train': [], 'val': []}

    for epoch in range(1, cfg['epochs'] + 1):
        # ── Train ──────────────────────────────────────
        model.train()
        tr_loss = 0.0
        for xb, yb in tr_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(xb), yb)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            tr_loss += loss.item()
        tr_loss /= len(tr_loader)

        # ── Validate ───────────────────────────────────
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for xb, yb in v_loader:
                xb, yb = xb.to(DEVICE), yb.to(DEVICE)
                val_loss += criterion(model(xb), yb).item()
        val_loss /= len(v_loader)

        scheduler.step(val_loss)
        history['train'].append(tr_loss)
        history['val'].append(val_loss)

        if val_loss < best_val_loss - 1e-5:
            best_val_loss = val_loss
            bad_epochs    = 0
            best_epoch    = epoch
            torch.save(model.state_dict(), ckpt_path)
        else:
            bad_epochs += 1

        if epoch % 10 == 0 or epoch == 1 or bad_epochs >= cfg['es_patience']:
            lr_now = optimizer.param_groups[0]['lr']
            print(f"    [{task_name}] ep{epoch:4d}  tr={tr_loss:.5f}  "
                  f"val={val_loss:.5f}  lr={lr_now:.2e}  "
                  f"best_ep={best_epoch}", flush=True)

        if bad_epochs >= cfg['es_patience']:
            print(f"    ⏹ Early stop at epoch {epoch}  "
                  f"(best={best_val_loss:.5f} @ ep{best_epoch})")
            break

    return ckpt_path, history, best_val_loss

# ── Decide task specs and get correct X arrays for each task ─────────────────
TASK_SPECS = [
    (task, CLASS_IDX[task],
     TASK_WINDOWS[task]['Xtr'],
     TASK_WINDOWS[task]['Xv'],
     TASK_WINDOWS[task]['Xte'])
    for task in TRAIN_TASKS
]

ALL_CKPTS   = {}
ALL_HISTORY = {}

for task_name, cls_idx, Xtr, Xva, Xte in TASK_SPECS:
    print(f"\n{'='*60}")
    print(f"🎯 Task: {task_name}  |  n_ch={Xtr.shape[1]}")
    print(f"{'='*60}")

    n_ch = Xtr.shape[1]

    Y_tr = TASK_WINDOWS[task_name]['Ytr'][:, cls_idx, :]
    Y_v  = TASK_WINDOWS[task_name]['Yv'][:,  cls_idx, :]

    pos_freq = float(Y_tr.mean())
    print(f"   n_ch={n_ch}  pos_freq={pos_freq:.4f}  train_n={len(Y_tr)}")

    task_ckpts   = []
    task_history = []

    for seed in CONFIG['seeds']:
        print(f"\n  Seed {seed}:")

        ckpt_path, hist, bvl = train_one_model(
            Xtr, Y_tr, Xva, Y_v,
            n_ch=n_ch, pos_freq=pos_freq,
            task_name=task_name,
            seed=seed,
            cfg=CONFIG
        )
        task_ckpts.append(ckpt_path)
        task_history.append(hist)
        print(f"  ✅ Seed {seed} done | best_val_loss={bvl:.5f}")

    ALL_CKPTS[task_name]   = task_ckpts
    ALL_HISTORY[task_name] = task_history

# ── Save training histories ──────────────────────────────────────────────
with open(HISTORY_PATH, 'w') as f:
    json.dump({k: [{'train': h['train'], 'val': h['val']}
                   for h in v]
               for k, v in ALL_HISTORY.items()}, f, indent=2)

print("\n✅ All training complete. Histories saved.")



🎯 Task: F  |  n_ch=6
   n_ch=6  pos_freq=0.3375  train_n=2776

  Seed 42:
    [F] ep   1  tr=0.05531  val=0.05392  lr=1.00e-03  best_ep=1
    [F] ep  10  tr=0.04763  val=0.04990  lr=1.00e-03  best_ep=9
    [F] ep  20  tr=0.04711  val=0.04959  lr=1.00e-03  best_ep=18
    [F] ep  30  tr=0.04668  val=0.05132  lr=1.00e-03  best_ep=27
    [F] ep  40  tr=0.04668  val=0.04926  lr=1.00e-03  best_ep=35
    [F] ep  50  tr=0.04610  val=0.04946  lr=5.00e-04  best_ep=42
    [F] ep  60  tr=0.04580  val=0.05038  lr=2.50e-04  best_ep=42
    [F] ep  62  tr=0.04560  val=0.04964  lr=2.50e-04  best_ep=42
    ⏹ Early stop at epoch 62  (best=0.04872 @ ep42)
  ✅ Seed 42 done | best_val_loss=0.04872

  Seed 1:
    [F] ep   1  tr=0.05690  val=0.05416  lr=1.00e-03  best_ep=1
    [F] ep  10  tr=0.04748  val=0.05010  lr=1.00e-03  best_ep=5
    [F] ep  20  tr=0.04692  val=0.04935  lr=5.00e-04  best_ep=14
    [F] ep  30  tr=0.04628  val=0.04973  lr=1.25e-04  best_ep=14
    [F] ep  34  tr=0.04631  val=0.04937  lr=1

## 🎚️ Section 10 — Threshold Selection (VAL set only — no leakage)
> For each task and each seed, find the threshold that maximizes F1 on the **validation set**.  
> Thresholds are never touched by test set data.  
> Final threshold = median across seeds (robust to outlier seeds).


In [33]:
from scipy.signal import medfilt as scipy_medfilt

THRESHOLDS = {}   # task → median threshold across seeds
VAL_SCORES = {}   # task → list of (N_val, W) score arrays (one per seed)
# ── ADD THESE TWO FUNCTIONS at top of Section 10 cell ────────────────────

def predict_task(ckpt_path, X, n_ch, ch_idx, cfg):
    """Load checkpoint → run inference → return (N, W) score array."""
    model = build_model(cfg, n_ch, out_ch=1).to(DEVICE)
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    model.eval()
    dummy_Y = np.zeros((len(X), X.shape[2]), dtype=np.float32)
    loader  = make_loader(X, dummy_Y, shuffle=False)
    scores  = []
    with torch.no_grad():
        for xb, _ in loader:
            out = model(xb.to(DEVICE))          # (B, 1, W)
            scores.append(out[:, 0, :].cpu().numpy())
    return np.concatenate(scores, axis=0)       # (N, W)

def sweep_threshold(scores_flat, gt_flat, thresholds=None):
    """Find the threshold that maximises F1 on val set."""
    if thresholds is None:
        thresholds = np.linspace(0.1, 0.9, 17)
    best_th, best_f1 = 0.5, 0.0
    for th in thresholds:
        pred = (scores_flat > th).astype(int)
        tp = int(np.sum(pred & gt_flat))
        fp = int(np.sum(pred & (1 - gt_flat)))
        fn = int(np.sum((1 - pred) & gt_flat))
        p  = tp / (tp + fp + 1e-9)
        r  = tp / (tp + fn + 1e-9)
        f1 = 2 * p * r / (p + r + 1e-9)
        if f1 > best_f1:
            best_f1, best_th = f1, float(th)
    return best_th, best_f1
for task_name, cls_idx, Xtr, Xva, Xte in TASK_SPECS:
    n_ch  = Xva.shape[1]
    ch_all = list(range(n_ch))

    # ── Decompose into per-class integer indices always ──────────────────
    # dual_model=True  → cls_idx is already int (0 or 1), one task per loop
    # dual_model=False → cls_idx is slice(0,2), joint model; sweep each class
    #                    separately because predict_task returns (N,W) not (N,2,W)


    sub_tasks = [(task_name, cls_idx)]   # always — joint mode not supported for 5-class

    for sub_task_name, sub_cls_idx in sub_tasks:
        print(f"\n🔍 Threshold sweep: task={sub_task_name}")

        gt_val_flat = TASK_WINDOWS[task_name]['Yv'][:, sub_cls_idx, :].reshape(-1).astype(int)

        seed_ths    = []
        seed_scores = []

        for seed_i, ckpt_path in enumerate(ALL_CKPTS[task_name]):
            scores_2d   = predict_task(ckpt_path, Xva, n_ch, ch_all, CONFIG)
            scores_flat = scores_2d.reshape(-1)
            th, val_f1  = sweep_threshold(scores_flat, gt_val_flat)
            seed_ths.append(th)
            seed_scores.append(scores_2d)
            print(f"   seed={CONFIG['seeds'][seed_i]}  th={th:.2f}  val_F1={val_f1:.4f}")

        final_th = float(np.median(seed_ths))
        THRESHOLDS[sub_task_name] = final_th
        VAL_SCORES[sub_task_name] = seed_scores
        print(f"   → Final threshold (median): {final_th:.2f}")

print("\n✅ All thresholds locked from val set.")
print("   THRESHOLDS:", THRESHOLDS)


🔍 Threshold sweep: task=F
   seed=42  th=0.45  val_F1=0.6805
   seed=1  th=0.45  val_F1=0.6792
   seed=2  th=0.45  val_F1=0.6788
   → Final threshold (median): 0.45

🔍 Threshold sweep: task=AV
   seed=42  th=0.40  val_F1=0.3904
   seed=1  th=0.35  val_F1=0.3588
   seed=2  th=0.40  val_F1=0.4319
   → Final threshold (median): 0.40

✅ All thresholds locked from val set.
   THRESHOLDS: {'F': 0.45000000000000007, 'AV': 0.4}


**TO CHANGE THRESHOLD MANUALLY**

In [34]:
# ── TEMPORARY THRESHOLD OVERRIDE ─────────────────────────────
# Original thresholds from val-sweep (Section 10):
#   K=0.75, AV=0.80, SAH=0.90  ← these feel too high

#THRESHOLDS['K']   = 0.40   # was 0.75
#THRESHOLDS['AV']  = 0.40   # was 0.80
#THRESHOLDS['SAH'] = 0.40   # was 0.90

#print("Overridden THRESHOLDS:", THRESHOLDS)
# ─────────────────────────────────────────────────────────────

## 📊 Section 11 — Final Evaluation on Test Set
> Thresholds come exclusively from Section 10 (val set).  
> Post-processing: median filter (kernel=11) removes sub-100ms spurious activations.  
> Three metric variants (paper-aligned):  
> 1. Frame-level, no collar  
> 2. Frame-level, 80ms collar  
> 3. Event-level, 200ms collar  
> 95% bootstrap CIs on all F1 scores.  
> Results averaged across seeds (mean ± std).


In [35]:
from scipy.ndimage import binary_dilation
from sklearn.metrics import f1_score, precision_score, recall_score

COLLAR_FRAME = int(round(0.080 / TIME_STEP))   # 8 frames  = 80ms
COLLAR_EVENT = int(round(0.200 / TIME_STEP))   # 20 frames = 200ms

# ── Re-define TASK_SPECS_EVAL as flat per-class 4-tuples ─────────────────
# Each entry: (sub_task_name, cls_idx_int, feat_key, ckpt_task_name)
# ckpt_task_name → key into ALL_CKPTS (always the original training task name)
# sub_task_name  → key into THRESHOLDS (matches what sweep cell stored)
TASK_SPECS_EVAL = [
    (task, CLASS_IDX[task], CONFIG['task_feature_map'][task], task)
    for task in TRAIN_TASKS
]
# Format: (sub_task_name, cls_idx, feat_key, ckpt_task_name)
# Map feat_key → correct standardized test array
Xte_map = {
    feat_key: TASK_WINDOWS[task]['Xte']
    for task, feat_key in CONFIG['task_feature_map'].items()
}


def events_from_binary(y):
    y    = np.asarray(y).astype(int)
    diff = np.diff(np.concatenate(([0], y, [0])))
    return list(zip(np.where(diff == 1)[0], np.where(diff == -1)[0]))

def frame_metrics(y_true, y_pred, collar_frames=0):
    y_true, y_pred = np.asarray(y_true).astype(bool), np.asarray(y_pred).astype(bool)
    if collar_frames > 0:
        y_true_d = binary_dilation(y_true, iterations=collar_frames)
        y_pred_d = binary_dilation(y_pred, iterations=collar_frames)
    else:
        y_true_d, y_pred_d = y_true, y_pred
    TP = int(np.sum(y_pred & y_true_d))
    FP = int(np.sum(y_pred & ~y_true_d))
    FN = int(np.sum(y_true & ~y_pred_d))
    P  = TP / (TP + FP) if (TP + FP) else 0.0
    R  = TP / (TP + FN) if (TP + FN) else 0.0
    F1 = 2*P*R / (P + R) if (P + R) else 0.0
    return P, R, F1, TP, FP, FN

def event_metrics(y_true, y_pred, collar_frames):
    gt, pr = events_from_binary(y_true), events_from_binary(y_pred)
    matched_pr, matched_gt = set(), set()
    for i, (gs, ge) in enumerate(gt):
        for j, (ps, pe) in enumerate(pr):
            if j in matched_pr: continue
            if ps <= ge + collar_frames and pe >= gs - collar_frames:
                matched_gt.add(i); matched_pr.add(j); break
    TP = len(matched_pr); FP = len(pr)-TP; FN = len(gt)-len(matched_gt)
    P  = TP / (TP + FP) if (TP + FP) else 0.0
    R  = TP / (TP + FN) if (TP + FN) else 0.0
    F1 = 2*P*R / (P + R) if (P + R) else 0.0
    return P, R, F1, TP, FP, FN

def bootstrap_f1_ci(gt, pred, n=1000, alpha=0.95):
    rng = np.random.default_rng(0)
    n_samples = len(gt)
    f1s = []
    for _ in range(n):
        idx = rng.integers(0, n_samples, size=n_samples)
        g_b, p_b = gt[idx], pred[idx]
        tp = int(np.sum(p_b & g_b))
        fp = int(np.sum(p_b & ~g_b))
        fn = int(np.sum(~p_b & g_b))
        p  = tp / (tp + fp + 1e-9)
        r  = tp / (tp + fn + 1e-9)
        f1s.append(2*p*r/(p+r+1e-9))
    lo = np.percentile(f1s, (1-alpha)/2*100)
    hi = np.percentile(f1s, (1+alpha)/2*100)
    return float(lo), float(hi)

# ── Get test predictions (keyed by ckpt_task_name, deduplicated) ──────────
TEST_SCORES = {}
for sub_task_name, cls_idx, feat_key, ckpt_task_name in TASK_SPECS_EVAL:
    if ckpt_task_name in TEST_SCORES:
        continue   # already computed for this model (joint mode runs twice)
    ch_idx = CHANNEL_MAP[feat_key]
    n_ch   = len(ch_idx)
    ch_all = list(range(n_ch))
    seed_scores = []
    for ckpt_path in ALL_CKPTS[ckpt_task_name]:
        scores_2d = predict_task(ckpt_path, Xte_map[feat_key], n_ch, ch_all, CONFIG)
        seed_scores.append(scores_2d)
    TEST_SCORES[ckpt_task_name] = seed_scores

# ── Evaluate ──────────────────────────────────────────────────────────────
print(f"\n{'='*80}")
print(f"  FINAL TEST SET EVALUATION   exp_id={CONFIG['exp_id']}")
print(f"  Thresholds (from VAL): {THRESHOLDS}")
print(f"{'='*80}")

all_results = {}

for sub_task_name, cls_idx, feat_key, ckpt_task_name in TASK_SPECS_EVAL:
    th      = THRESHOLDS[sub_task_name]          # ✅ sub_task_name matches sweep cell
    gt_flat = TASK_WINDOWS[ckpt_task_name]['Yte'][:, cls_idx, :].reshape(-1).astype(bool)

    seed_results = {'frame_nocol': [], 'frame_col80': [], 'event_col200': []}

    for seed_i, scores_2d in enumerate(TEST_SCORES[ckpt_task_name]):
        scores_flat = scores_2d.reshape(-1)
        pred_raw    = (scores_flat > th).astype(np.float32)
        pred_bin    = scipy_medfilt(pred_raw, kernel_size=CONFIG['medfilt_kernel']).astype(bool)

        P, R, F1, TP, FP, FN = frame_metrics(gt_flat, pred_bin, collar_frames=0)
        seed_results['frame_nocol'].append((P, R, F1, TP, FP, FN))

        P, R, F1, TP, FP, FN = frame_metrics(gt_flat, pred_bin, collar_frames=COLLAR_FRAME)
        seed_results['frame_col80'].append((P, R, F1, TP, FP, FN))

        P, R, F1, TP, FP, FN = event_metrics(gt_flat, pred_bin, collar_frames=COLLAR_EVENT)
        seed_results['event_col200'].append((P, R, F1, TP, FP, FN))

    print(f"\n── Task: {sub_task_name}  [threshold={th:.2f}] ──")
    print(f"  {'Metric':<24} | {'P':>6} | {'R':>6} | {'F1':>6} | {'F1 std':>7} | {'95% CI (F1)':<20}")
    print(f"  {'-'*75}")

    task_summary = {}
    label_map = {
        'frame_nocol' : 'frame, no collar',
        'frame_col80' : 'frame, collar 80ms',
        'event_col200': 'event, collar 200ms'
    }
    for metric_name, results_list in seed_results.items():
        F1s = np.array([r[2] for r in results_list])
        Ps  = np.array([r[0] for r in results_list])
        Rs  = np.array([r[1] for r in results_list])
        mean_F1, std_F1 = float(F1s.mean()), float(F1s.std())
        ci_lo, ci_hi    = float(F1s.min()), float(F1s.max())
        print(f"  {label_map[metric_name]:<24} | "
              f"{Ps.mean():>6.3f} | {Rs.mean():>6.3f} | "
              f"{mean_F1:>6.3f} | ±{std_F1:>5.3f}  | "
              f"[{ci_lo:.3f}, {ci_hi:.3f}]")
        task_summary[metric_name] = {
            'P_mean':  round(float(Ps.mean()), 4),
            'R_mean':  round(float(Rs.mean()), 4),
            'F1_mean': round(mean_F1, 4),
            'F1_std':  round(std_F1, 4),
            'CI_lo':   round(ci_lo, 4),
            'CI_hi':   round(ci_hi, 4),
        }

    n_pos_frames = int(gt_flat.sum())
    n_tot_frames = len(gt_flat)
    print(f"\n  Positive frames: {n_pos_frames:,} / {n_tot_frames:,} "
          f"({n_pos_frames/n_tot_frames*100:.2f}%)")

    all_results[sub_task_name] = task_summary

print(f"\n{'='*80}")

# ── Save results ──────────────────────────────────────────────────────────
final_out = {
    'exp_id':     CONFIG['exp_id'],
    'config':     CONFIG,
    'thresholds': THRESHOLDS,
    'seeds':      CONFIG['seeds'],
    'metrics':    all_results,
    'n_params':   {
        ckpt_task_name: sum(
            p.numel() for p in build_model(
                CONFIG, len(CHANNEL_MAP[feat_key]), 1
            ).parameters()
        )
        for sub_task_name, _, feat_key, ckpt_task_name in TASK_SPECS_EVAL
    }
}
with open(RESULTS_PATH, 'w') as f:
    json.dump(final_out, f, indent=2)
print(f"\n💾 Results saved → {RESULTS_PATH}")


  FINAL TEST SET EVALUATION   exp_id=E3_focal_fixed
  Thresholds (from VAL): {'F': 0.45000000000000007, 'AV': 0.4}

── Task: F  [threshold=0.45] ──
  Metric                   |      P |      R |     F1 |  F1 std | 95% CI (F1)         
  ---------------------------------------------------------------------------
  frame, no collar         |  0.528 |  0.861 |  0.655 | ±0.003  | [0.650, 0.659]
  frame, collar 80ms       |  0.559 |  0.885 |  0.686 | ±0.003  | [0.681, 0.689]
  event, collar 200ms      |  0.639 |  0.715 |  0.674 | ±0.015  | [0.656, 0.692]

  Positive frames: 99,833 / 282,800 (35.30%)

── Task: AV  [threshold=0.40] ──
  Metric                   |      P |      R |     F1 |  F1 std | 95% CI (F1)         
  ---------------------------------------------------------------------------
  frame, no collar         |  0.277 |  0.203 |  0.234 | ±0.060  | [0.180, 0.318]
  frame, collar 80ms       |  0.282 |  0.210 |  0.241 | ±0.061  | [0.185, 0.325]
  event, collar 200ms      |  0.204 

## 📏 Section 12 — Rule-Based Baseline
> Hard to beat a well-tuned rule baseline in small datasets.  
> Provides a sanity-check floor: if the DL model doesn't beat this, something is wrong.
>
> **F-fault rule:** `circular_dist(pitch_s, pitch_t) > β_F` for sustained frames  
> **A-fault rule:** `|energy_s - energy_t| > β_A` for sustained frames  
> Thresholds β_F and β_A are grid-searched on the **val set** only.


In [36]:
from itertools import product as iproduct

# ── Features for the rule baseline (use raw unstandardized values) ────────
# ch2 = pitch_diff (circular), ch5 = energy_diff (linear)
# We use the base6 feature set directly from raw NPZ data

def rule_predict_flat(X_std_arr, task, th):
    """
    X_std_arr: (N, 6, W) standardized — but we re-use relative channels
    task: 'F' → use channel 2 (pitch_diff); 'A' → use channel 5 (energy_diff)
    Returns flat binary prediction.
    """
    if task == 'F':
        # channel index depends on current feature config
        # For rule baseline always use base6 regardless of model config
        pass
    # We'll compute separately from raw data below
    pass


# Use raw (unstandardized) test & val data for rule baseline
# Re-load if needed (we can use X_val and X_test before standardization)
# The circular diff is channel 2 in base6; energy diff is channel 5

def extract_rule_channels(X_raw, feature_key):
    """Extract pitch_diff and energy_diff from raw 6-ch data."""
    # These are always indices 2 and 5 in base6 superset
    # For non-base6 configs we still need these — reload raw base6
    return X_raw   # caller picks channels


# Re-window with base6 to get rule-baseline channels
print("📂 Loading base6 windows for rule baseline...")
X_tr_raw6, Y_tr_6 = load_and_window(train_fnames, 'base6', include_aug=False, desc="Rule-tr")
X_v_raw6,  Y_v_6  = load_and_window(val_fnames,   'base6', include_aug=False, desc="Rule-val")
X_te_raw6, Y_te_6 = load_and_window(test_fnames,  'base6', include_aug=False, desc="Rule-test")

# pitch_diff = ch 2, energy_diff = ch 5
PDIFF_VAL  = X_v_raw6[:,  2, :].reshape(-1)   # (N*W,)
EDIFF_VAL  = X_v_raw6[:,  5, :].reshape(-1)
PDIFF_TEST = X_te_raw6[:, 2, :].reshape(-1)
EDIFF_TEST = X_te_raw6[:, 5, :].reshape(-1)

GT_F_VAL   = Y_v_6[:,  0, :].reshape(-1).astype(int)
GT_AV_VAL  = Y_v_6[:,  3, :].reshape(-1).astype(int)   # AV at idx 3
GT_F_TEST  = Y_te_6[:, 0, :].reshape(-1).astype(int)
GT_AV_TEST = Y_te_6[:, 3, :].reshape(-1).astype(int)   # AV at idx 3

print("\n🔍 Grid search rule thresholds on VAL set...")
best_bF, best_bA = 0.05, 0.5
best_f1F, best_f1A = 0.0, 0.0

for bF in np.arange(0.05, 0.50, 0.05):
    pred_F = (np.abs(PDIFF_VAL) > bF).astype(int)
    pred_F = scipy_medfilt(pred_F.astype(float), kernel_size=11).astype(int)
    tp = int(np.sum(pred_F & GT_F_VAL))
    fp = int(np.sum(pred_F & (1 - GT_F_VAL)))
    fn = int(np.sum((1 - pred_F) & GT_F_VAL))
    p  = tp / (tp+fp+1e-9); r = tp / (tp+fn+1e-9)
    f1 = 2*p*r/(p+r+1e-9)
    if f1 > best_f1F: best_f1F, best_bF = f1, float(bF)

for bA in np.arange(0.2, 3.0, 0.1):
    pred_A = (np.abs(EDIFF_VAL) > bA).astype(int)
    pred_A = scipy_medfilt(pred_A.astype(float), kernel_size=11).astype(int)
    tp = int(np.sum(pred_A & GT_AV_VAL))
    fp = int(np.sum(pred_A & (1-GT_AV_VAL)))
    fn = int(np.sum((1-pred_A) & GT_AV_VAL))
    p  = tp / (tp+fp+1e-9); r = tp / (tp+fn+1e-9)
    f1 = 2*p*r/(p+r+1e-9)
    if f1 > best_f1A: best_f1A, best_bA = f1, float(bA)

print(f"   Best β_F={best_bF:.2f}  val_F1_F={best_f1F:.4f}")
print(f"   Best β_A={best_bA:.2f}  val_F1_A={best_f1A:.4f}")

# ── Apply to test set ─────────────────────────────────────────────────────
print("\n📊 RULE BASELINE — Test set results:")
print(f" {'Metric':<24} | {'Task':^5} | {'P':>6} | {'R':>6} | {'F1':>6}")
print(f" {'-'*54}")

baseline_results = {}
for task, diff_flat, gt_flat, beta in [
    ('F', PDIFF_TEST, GT_F_TEST, best_bF),
    ('AV', EDIFF_TEST, GT_AV_TEST, best_bA),
]:
    pred_raw = (np.abs(diff_flat) > beta).astype(int)
    pred_bin = scipy_medfilt(pred_raw.astype(float), kernel_size=11).astype(bool)
    gt_bool  = gt_flat.astype(bool)

    for metric_name, collar in [('frame, no collar', 0),
                                  ('frame, col 80ms', COLLAR_FRAME),
                                  ('event, col 200ms', COLLAR_EVENT)]:
        if 'event' in metric_name:
            P, R, F1, *_ = event_metrics(gt_bool, pred_bin, collar)
        else:
            P, R, F1, *_ = frame_metrics(gt_bool, pred_bin, collar)

        print(f" {metric_name:<24} | {task:^5} | {P:>6.3f} | {R:>6.3f} | {F1:>6.3f}")
        baseline_results[f"{task}_{metric_name.replace(' ','_')}"] = {
            'P': round(P,4), 'R': round(R,4), 'F1': round(F1,4)
        }
        

# ── Save baseline results ─────────────────────────────────────────────────
baseline_path = os.path.join(EXP_DIR, "baseline_results.json")
with open(baseline_path, 'w') as f:
    json.dump({'beta_F': best_bF, 'beta_A': best_bA,
               'metrics': baseline_results}, f, indent=2)
print(f"\n💾 Baseline results saved → {baseline_path}")


📂 Loading base6 windows for rule baseline...


Rule-test: 100%|██████████| 26/26 [00:00<00:00, 627.36it/s]



🔍 Grid search rule thresholds on VAL set...
   Best β_F=0.05  val_F1_F=0.4786
   Best β_A=0.30  val_F1_A=0.1156

📊 RULE BASELINE — Test set results:
 Metric                   | Task  |      P |      R |     F1
 ------------------------------------------------------
 frame, no collar         |   F   |  0.563 |  0.433 |  0.489
 frame, col 80ms          |   F   |  0.588 |  0.564 |  0.576
 event, col 200ms         |   F   |  0.142 |  0.963 |  0.248
 frame, no collar         |  AV   |  0.026 |  0.377 |  0.049
 frame, col 80ms          |  AV   |  0.029 |  0.482 |  0.055
 event, col 200ms         |  AV   |  0.013 |  1.000 |  0.025

💾 Baseline results saved → C:\Users\RICHIK\Desktop\Monesh\Music_Work\experimentation_final_3_new_annotations\experiments\E3_focal_fixed\baseline_results.json


## 🏆 Section 13 — Ablation Summary Table
> Run this cell **after all experiments** to generate the comparison table.  
> Reads every `eval_results.json` and `baseline_results.json` in the experiments folder.


In [37]:
EXP_ROOT = os.path.join(BASE_DIR, "experiments")
rows = []

for exp_name in sorted(os.listdir(EXP_ROOT)):
    result_file = os.path.join(EXP_ROOT, exp_name, "eval_results.json")
    if not os.path.exists(result_file):
        continue
    with open(result_file) as f:
        r = json.load(f)
    m = r.get('metrics', {})
    cfg = r.get('config', {})

    def get(task, metric, key):
        return m.get(task, {}).get(metric, {}).get(key, '-')

    row = {
        'Exp':           r['exp_id'],
        'Model':         cfg.get('model', '-'),
        'Loss':          cfg.get('loss', '-'),
        'Feats':         cfg.get('features', '-'),
        'DualModel':     cfg.get('dual_model', '-'),
        'Aug':           cfg.get('augment_A', '-'),
        'Split':         cfg.get('split', '-'),
        'Seeds':         str(cfg.get('seeds', '-')),
    }

    # Primary metrics: frame-level 80ms collar (paper's main metric)
    for task in TRAIN_TASKS:
        for metric_key, label in [('frame_col80', 'col80'), ('event_col200', 'ev200')]:
            f1    = get(task, metric_key, 'F1_mean')
            f1std = get(task, metric_key, 'F1_std')
            ci_lo = get(task, metric_key, 'CI_lo')
            ci_hi = get(task, metric_key, 'CI_hi')
            if f1 != '-':
                row[f'{task}_F1_{label}'] = f"{f1:.3f}±{f1std:.3f}"
                row[f'{task}_CI_{label}'] = f"[{ci_lo:.3f},{ci_hi:.3f}]"

    rows.append(row)

if not rows:
    print("⚠️  No results found yet. Run at least one experiment first.")
else:
    df_summary = pd.DataFrame(rows)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', 200)
    print(df_summary.to_string(index=False))
    out_csv = os.path.join(EXP_ROOT, "ablation_summary.csv")
    df_summary.to_csv(out_csv, index=False)
    print(f"\n💾 Ablation table saved → {out_csv}")


                 Exp    Model  Loss Feats  DualModel   Aug     Split      Seeds  F_F1_col80    F_CI_col80  F_F1_ev200    F_CI_ev200 AV_F1_col80   AV_CI_col80 AV_F1_ev200   AV_CI_ev200
         E0_baseline     crnn  wbce base6       True False recording [42, 1, 2] 0.677±0.002 [0.675,0.681] 0.680±0.006 [0.672,0.686] 0.281±0.018 [0.257,0.301] 0.292±0.006 [0.283,0.298]
   E0_baseline_fixed     crnn  wbce base6       True False recording [42, 1, 2] 0.673±0.005 [0.667,0.678] 0.645±0.006 [0.637,0.650] 0.271±0.034 [0.227,0.310] 0.275±0.008 [0.264,0.284]
              E1_tcn      tcn  wbce base6       True False recording [42, 1, 2] 0.665±0.006 [0.660,0.674] 0.680±0.020 [0.654,0.702] 0.280±0.018 [0.263,0.306] 0.254±0.005 [0.249,0.261]
          E2_deepcnn deep_cnn  wbce base6       True False recording [42, 1, 2] 0.656±0.004 [0.652,0.661] 0.594±0.015 [0.580,0.615] 0.222±0.007 [0.214,0.231] 0.158±0.020 [0.139,0.185]
            E3_focal     crnn focal base6       True False recording [42, 1, 2] 

## 📋 Section 14 — Per-Student Breakdown (Appendix Table)
> Reports performance broken down per student — required for paper appendix.  
> Exposes whether the model works better for some students than others.


In [ ]:
print("📊 Per-student breakdown on test recordings:")
print(f"  {'Student':^12} | {'Task':^5} | {'F1 frame_col80':>14} | {'n_pos_frames':>12}")
print(f"  {'-'*55}")

for student_id in sorted(set(all_student_ids)):
    # Find test recordings for this student
    stu_fnames = [
        f for i, f in enumerate(all_fnames)
        if all_student_ids[i] == student_id and f in test_fnames
    ]
    if not stu_fnames:
        continue

    # Window once for labels (same regardless of task feature subset)
    _, Y_stu = load_and_window(stu_fnames, 'base6', include_aug=False, desc=f"  {student_id}")
    if len(Y_stu) == 0:
        continue

    # Build standardized X per feature key once (deduplicated)
    Xstu_map = {}
    for task in TRAIN_TASKS:
        feat_key = CONFIG['task_feature_map'][task]
        if feat_key in Xstu_map:
            continue

        X_stu_t, _ = load_and_window(stu_fnames, feat_key, include_aug=False)
        if len(X_stu_t) == 0:
            continue

        Xstu_map[feat_key] = TASK_SCALERS[task].transform(X_stu_t)

    # Evaluate all tasks for this student
    for sub_task_name, cls_idx, feat_key, ckpt_task_name in TASK_SPECS_EVAL:
        if feat_key not in Xstu_map:
            print(f"  ⚠️  {student_id}: missing windows for {sub_task_name} ({feat_key}), skipping")
            continue

        # ✅ Define ground truth for this student/task
        gt_stu = Y_stu[:, cls_idx, :].reshape(-1).astype(bool)

        th = THRESHOLDS[sub_task_name]
        n_ch = Xstu_map[feat_key].shape[1]   # relative to already-sliced array
        ch_idx = list(range(n_ch))           # relative indices 0..n_ch-1

        # Average predictions across seeds
        all_seed_scores = []
        for ckpt_path in ALL_CKPTS[ckpt_task_name]:
            sc = predict_task(ckpt_path, Xstu_map[feat_key], n_ch, ch_idx, CONFIG)
            all_seed_scores.append(sc.reshape(-1))

        avg_sc = np.mean(all_seed_scores, axis=0)
        pred_bin = scipy_medfilt(
            (avg_sc > th).astype(np.float32),
            kernel_size=CONFIG['medfilt_kernel']
        ).astype(bool)

        # Safety: align lengths if needed
        if len(pred_bin) != len(gt_stu):
            n = min(len(pred_bin), len(gt_stu))
            print(f"  ⚠️  {student_id}/{sub_task_name}: len mismatch pred={len(pred_bin)} gt={len(gt_stu)}; trimming to {n}")
            pred_eval = pred_bin[:n]
            gt_eval = gt_stu[:n]
        else:
            pred_eval = pred_bin
            gt_eval = gt_stu

        _, _, F1, TP, FP, FN = frame_metrics(gt_eval, pred_eval, collar_frames=COLLAR_FRAME)
        n_pos = int(gt_eval.sum())
        print(f"  {student_id:^12} | {sub_task_name:^5} | {F1:>14.4f} | {n_pos:>12,}")

📊 Per-student breakdown on test recordings:
    Student    | Task  | F1 frame_col80 | n_pos_frames
  -------------------------------------------------------


Windowing: 100%|██████████| 10/10 [00:00<00:00, 1428.58it/s]

     s_2301    |   F   |         0.7765 |       35,241


     s_2301    |  AV   |         0.0000 |          246


Windowing: 100%|██████████| 11/11 [00:00<00:00, 916.08it/s]

     s_2302    |   F   |         0.5752 |       38,364


     s_2302    |  AV   |         0.2645 |        9,204


Windowing: 100%|██████████| 5/5 [00:00<00:00, 999.88it/s]

     s_2303    |   F   |         0.7862 |       26,228
     s_2303    |  AV   |         0.0000 |           92


: 

---
## 🗂️ Output Files in `experimentation_final_2/`

```
experimentation_final_2/
├── npz_files/
│   ├── sample_000.npz           ← 6-ch features + frame labels per recording
│   └── sample_000_aug.npz       ← synthetic A-fault augmented version
├── splits/
│   ├── recording_split_filenames.json   ← recording-level split (fixed)
│   └── loso_split_<id>_filenames.json  ← LOSO split (per student)
└── experiments/
    └── E0_corrected_baseline/
        ├── config.json
        ├── scaler_stats.npz             ← channel mean/std (train only)
        ├── model_F_seed42.pt            ← best F-model checkpoint (seed 42)
        ├── model_F_seed1.pt
        ├── model_F_seed2.pt
        ├── model_A_seed42.pt            ← best A-model checkpoint (seed 42)
        ├── model_A_seed1.pt
        ├── model_A_seed2.pt
        ├── training_history.json
        ├── eval_results.json            ← all metrics with CI
        └── baseline_results.json        ← rule-based baseline metrics
```
